# 03 - Feature Engineering

**Objective:** Turn the raw columns into numeric inputs a linear model can use, and *test* (not assume) whether extra engineered features help.

**Role in the ML workflow:** *Feature engineering* - prepare and improve the input features.

### Concepts to understand first
* **Preprocessing vs feature engineering**
  * *Preprocessing* makes existing data usable: encoding text categories as numbers, scaling numbers. Example: `smoker = "yes"` -> `1`.
  * *Feature engineering* creates **new** columns that expose a pattern the model could not otherwise see. Example: `smoker_bmi = bmi if smoker else 0`, because EDA showed BMI only matters for smokers.
* **One-hot encoding:** a category with k levels becomes k-1 columns of 0/1 (one level is dropped as the reference). `region` -> `region_northwest`, `region_southeast`, `region_southwest`; northeast is the reference (all zeros). Use it for *nominal* categories - those with no natural order.
* **Binary encoding:** for a two-level variable one 0/1 column is enough (`smoker_yes`).
* **Why not `northeast=0, northwest=1, southeast=2, southwest=3`?** That invents an order and equal spacing ("southwest is 3x northwest") that does not exist.
* **Data leakage:** using information from the test set while preparing the model. Anything that *learns from data* (scaler means, encoder categories) must be fitted on **training data only**. We put every such step inside a scikit-learn `Pipeline`, so it is refitted on the training portion of each cross-validation fold automatically.
* **Cross-validation (CV):** split the *training* data into 5 folds, train on 4 and score on the 5th, rotate, and average. It gives a performance estimate without touching the test set.

> The test set stays locked away in this notebook. All comparisons use CV on the training set only.

## Setup and split
The split is made **before** any transformation is fitted (full explanation in notebook 04).

In [1]:
import sys
import warnings
from pathlib import Path

# Make the project root importable so we can use the code in src/.
# (Jupyter's working directory is notebooks/, so the project root is one level up.)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

from src.data_preprocessing import load_clean_data, split_data
from src.feature_engineering import (
    add_engineered_features, build_preprocessor, ALL_ENGINEERED,
)
from src.train import CV

df = load_clean_data()
X_train, X_test, y_train, y_test = split_data(df)
print("Training rows:", len(X_train), "| Test rows (not used here):", len(X_test))

def cv_rmse(model):
    """Mean 5-fold CV RMSE on the TRAINING data (lower is better)."""
    scores = cross_val_score(model, X_train, y_train, cv=CV, scoring="neg_root_mean_squared_error")
    return -scores.mean()

Training rows: 1069 | Test rows (not used here): 268


## Part A - Encoding the original features

The pipeline built by `src/feature_engineering.py` encodes `sex` and `smoker` as single 0/1 columns, one-hot encodes `region`, and keeps `age`, `bmi`, `children` numeric. Look at exactly what the model receives:

In [3]:
pre = build_preprocessor(engineered=(), scale_numeric=False)
encoded = pre.fit(X_train).transform(X_train)
feature_names = pre.named_steps["encode_scale"].get_feature_names_out()
print("Encoded columns:", list(feature_names))
pd.DataFrame(encoded, columns=feature_names).head()

Encoded columns: ['numeric__age', 'numeric__bmi', 'numeric__children', 'binary__sex_male', 'binary__smoker_yes', 'multi__region_northwest', 'multi__region_southeast', 'multi__region_southwest']


,numeric__age,numeric__bmi,numeric__children,binary__sex_male,binary__smoker_yes,multi__region_northwest,multi__region_southeast,multi__region_southwest
0,23.0,24.510,0.0,1.0,0.0,0.0,0.0,0.0
1,21.0,25.745,2.0,1.0,0.0,0.0,0.0,0.0
2,52.0,37.525,2.0,0.0,0.0,1.0,0.0,0.0
3,63.0,41.470,0.0,1.0,0.0,0.0,1.0,0.0
4,47.0,26.600,2.0,0.0,0.0,0.0,0.0,0.0


*Reading it:* the `numeric__` columns are unchanged numbers, `binary__sex_male` and `binary__smoker_yes` are 0/1, and the `multi__region_...` columns are the three region dummies. One input row becomes 8 numbers.

### Comparing encoding choices (OLS, 5-fold CV on training data)

In [4]:
def ols_with(numeric, binary=(), onehot=(), ordinal=()):
    parts = [("num", "passthrough", list(numeric))]
    if binary:  parts.append(("bin", OneHotEncoder(drop="if_binary"), list(binary)))
    if onehot:  parts.append(("oh",  OneHotEncoder(drop="first"), list(onehot)))
    if ordinal: parts.append(("ord", OrdinalEncoder(), list(ordinal)))
    return Pipeline([("prep", ColumnTransformer(parts)), ("model", LinearRegression())])

options = {
    "A. Recommended: one-hot region, binary sex/smoker, children numeric":
        ols_with(["age", "bmi", "children"], binary=["sex", "smoker"], onehot=["region"]),
    "B. Region as 0-3 integer codes (WRONG for nominal data)":
        ols_with(["age", "bmi", "children"], binary=["sex", "smoker"], ordinal=["region"]),
    "C. Children one-hot encoded instead of numeric":
        ols_with(["age", "bmi"], binary=["sex", "smoker"], onehot=["region", "children"]),
    "D. Drop sex and region":
        ols_with(["age", "bmi", "children"], binary=["smoker"]),
    "E. Drop smoker (to see how much it matters)":
        ols_with(["age", "bmi", "children"], binary=["sex"], onehot=["region"]),
}
encoding_results = pd.Series({name: cv_rmse(m) for name, m in options.items()}, name="CV RMSE").round(1)
encoding_results.to_frame()

,CV RMSE
"A. Recommended: one-hot region, binary sex/smoker, children numeric",6123.7
B. Region as 0-3 integer codes (WRONG for nominal data),6111.4
C. Children one-hot encoded instead of numeric,6130.2
D. Drop sex and region,6112.3
E. Drop smoker (to see how much it matters),11070.0


**What the encoding comparison shows (CV RMSE, lower is better)**
* **Smoker matters enormously:** removing it raises the error from about 6,124 to about 11,070 (E). It is by far the most useful feature.
* **Sex and region carry almost no signal:** dropping both (D) changes the error by only about 12, far below the fold-to-fold noise (the standard deviation across folds is roughly 200). We keep them in the baseline because the project asks us to investigate all six features and they cost almost nothing, but they are not doing much work.
* **Integer-coded region (B) did *not* score worse here** (about 6,111). That does not make it correct: the integer codes impose a fake order, and it only looks harmless because region is nearly irrelevant in this dataset. One-hot encoding is the safe default and would matter for a region that really did affect prices.
* **Children as a number (A) vs one-hot (C):** about 6,124 vs 6,130 - no real difference, so the simpler numeric treatment is justified for this data.

## Part B - Engineered features

We investigate five candidates, all rule-based (no statistics learned from data, so they cannot leak):

| Feature | Definition | Motivation from EDA |
|---|---|---|
| `bmi_category` | underweight (<18.5), normal, overweight (25-30), obese (>=30) | non-linear BMI effect near 30 |
| `age_group` | 18-29, 30-39, 40-49, 50+ | allows a non-straight age effect |
| `smoker_bmi` | `bmi` if smoker, else 0 | BMI matters mainly for smokers |
| `smoker_age` | `age` if smoker, else 0 | age slope may differ for smokers |
| `smoker_obese` | 1 if smoker **and** BMI >= 30, else 0 | the jump at BMI 30 seen in the scatterplot |

`smoker_obese` was not on the original list; it was added because the EDA scatterplot showed a clear step at BMI 30 for smokers. It is tested the same way as the others.

In [5]:
# What the new columns look like (first rows of the training set)
add_engineered_features(X_train).head()

,age,sex,bmi,children,smoker,region,bmi_category,age_group,smoker_bmi,smoker_age,smoker_obese
1113,23,male,24.510,0,no,northeast,normal,18-29,0.0,0,0
967,21,male,25.745,2,no,northeast,overweight,18-29,0.0,0,0
598,52,female,37.525,2,no,northwest,obese,50+,0.0,0,0
170,63,male,41.470,0,no,southeast,obese,50+,0.0,0,0
275,47,female,26.600,2,no,northeast,overweight,40-49,0.0,0,0


In [6]:
print(pd.cut(X_train["bmi"], [0, 18.5, 25, 30, np.inf], right=False).value_counts().sort_index())

bmi
[0.0, 18.5)      14
[18.5, 25.0)    185
[25.0, 30.0)    317
[30.0, inf)     553
Name: count, dtype: int64


### Does each feature actually improve the model? (ablation by CV)

In [7]:
feature_sets = {
    "None (baseline)":                          (),
    "+ bmi_category":                           ("bmi_category",),
    "+ age_group":                              ("age_group",),
    "+ bmi_category + age_group":               ("bmi_category", "age_group"),
    "+ smoker_bmi":                             ("smoker_bmi",),
    "+ smoker_age":                             ("smoker_age",),
    "+ smoker_obese":                           ("smoker_obese",),
    "+ smoker_bmi + smoker_age":                ("smoker_bmi", "smoker_age"),
    "+ smoker_bmi + smoker_age + smoker_obese": ("smoker_bmi", "smoker_age", "smoker_obese"),
    "+ ALL five":                               ALL_ENGINEERED,
}

rows = []
for name, feats in feature_sets.items():
    model = Pipeline([("pre", build_preprocessor(feats, scale_numeric=False)), ("model", LinearRegression())])
    rows.append({"Feature set": name, "CV RMSE": cv_rmse(model)})
ablation = pd.DataFrame(rows)
baseline = ablation.loc[0, "CV RMSE"]
ablation["Change vs baseline"] = (ablation["CV RMSE"] - baseline).round(1)
ablation["CV RMSE"] = ablation["CV RMSE"].round(1)
ablation

,Feature set,CV RMSE,Change vs baseline
0,None (baseline),6123.7,0.0
1,+ bmi_category,6054.5,-69.2
2,+ age_group,6076.1,-47.5
3,+ bmi_category + age_group,6016.2,-107.5
4,+ smoker_bmi,4943.0,-1180.6
5,+ smoker_age,6148.8,25.2
6,+ smoker_obese,4575.0,-1548.7
7,+ smoker_bmi + smoker_age,4951.0,-1172.6
8,+ smoker_bmi + smoker_age + smoker_obese,4513.5,-1610.2
9,+ ALL five,4480.2,-1643.5


**What the ablation shows (change in CV RMSE vs the baseline of about 6,124; negative = better)**

| Feature(s) | Verdict |
|---|---|
| `bmi_category`, `age_group` (alone or together) | small gains (about 50-110); modest but consistent |
| `smoker_age` | **no help** (slightly worse, +25) - the age slope is already similar for smokers and non-smokers once the level difference is captured |
| `smoker_bmi` | **large gain** (about -1,180) |
| `smoker_obese` | **largest single gain** (about -1,550) |
| `smoker_bmi + smoker_age + smoker_obese` | about -1,610 |
| all five | about -1,644 (CV RMSE about 4,480) - the best set, a 27% error reduction |

Take-aways: the improvement comes almost entirely from the **smoker interactions**, confirming what the scatterplots suggested. `smoker_obese` captures the step at BMI 30 more efficiently than a straight-line `smoker_bmi`. Adding `smoker_age` did not help and shows why we test instead of assuming. The extra gain from `bmi_category` and `age_group` once the interactions are present is small (about 30); we keep them in the candidate set and let the later model comparison and selection rule decide how much complexity is justified.

## Summary

* Encoding choices are tested with cross-validation, not assumed.
* All engineered features are rule-based and live inside the pipeline, so there is no target leakage and no information from the test set.
* The feature set carried forward is `ALL_ENGINEERED` as a candidate - the controlled five-model comparison is in notebook 06.

## Common errors and fixes
| Error | Cause | Fix |
|---|---|---|
| `ValueError: Found unknown categories` | prediction data has a label not seen in training (e.g. `"North"`) | validate/normalise input first (`validate_features` does this) |
| `KeyError` for `smoker_bmi` etc. | the engineered columns are only created inside the pipeline | pass the **raw** DataFrame to the pipeline; do not add columns by hand |
| Scores differ every run | CV splitter without a seed | we use `KFold(shuffle=True, random_state=42)` |

## Practical task
1. Add a feature `age_squared = age ** 2` as a new option in `add_engineered_features` (numeric) and test it in the ablation table. Does it help?
2. Explain in two sentences why we cannot compute the mean BMI on the full dataset and use it for scaling before splitting.

**Next:** notebook 04 - train/test split and OLS.